In [ ]:
import os
import sqlite3
import pandas as pd

In [ ]:
# print(os.getcwd())

In [ ]:
connection = sqlite3.connect("bdd")

cursor = connection.cursor()

In [ ]:
pd.read_sql("""
    SELECT name 
    FROM sqlite_master
    WHERE type = 'table'
""", connection)

- Ingresos por mes.
- Margen por categoría.
- Productos más vendidos.
- Clientes por país y canal.
- Tiempo medio de entrega.
- Tasa de cancelación o devolución.
- Rating medio por categoría o producto.[cite:1]

#### 1. INGRESOS POR MES

In [ ]:
cursor.execute("""
SELECT 
    DATE_FORMAT(o.delivered_date, '%Y-%m') AS mes,
    SUM(oi.quantity * oi.unit_price) AS ingresos_mes
FROM orders o
JOIN order_items oi 
    ON o.id = oi.order_id
JOIN payments p 
    ON o.id = p.order_id
WHERE 
    o.status = 'delivered'
    AND p.status = 'completed'
    AND p.payment_type = 'payment'
    AND MONTH(o.delivered_at) = MONTH(CURRENT_DATE())
    AND YEAR(o.delivered_at) = YEAR(CURRENT_DATE())
GROUP BY 
    DATE_FORMAT(o.delivered_at, '%Y-%m');""")


# para que muestre la descripción entera, si no se corta
# pd.set_option('display.max_colwidth', None)

df = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])


#### 2. MARGEN POR CATEGORÍA

In [ ]:
''' 
para calcular el margen tenemos que calcular el total de ingresos SUM(oi.quantity * oi.unit_price) y de ahí restar el total de costes SUM(oi.quantity * p.cost) AS costes.
Se calculará de pedidos que sí existen y están pagados y entregados. Para que el margen sea correcto la venta tiene que ser válida es decir_ :
- Para cada pedido tiene que tener líneas de pedido existentes (INNER JOIN order_items ) y el pedido tiene que estar realizado (WHERE o.status = 'delivered')
- Tiene que tener categorías existentes(INNER JOIN categories)
- Tiene que existir un pago (INNER JOIN payments pay) realizado (WHERE pay.status = 'completed' AND pay.payment_type = 'payment')
items que existen
Por último se agrupará por categorías
'''

cursor.execute("""
SELECT 
    c.name AS categoria,
    SUM(oi.quantity * oi.unit_price) AS ingresos,
    SUM(oi.quantity * p.cost) AS costes,
    SUM(oi.quantity * (oi.unit_price - p.cost)) AS margen
FROM orders o
INNER JOIN order_items oi 
    ON o.id = oi.order_id
INNER JOIN products p 
    ON oi.product_id = p.id
INNER JOIN categories c
    ON p.category_id = c.id
INNER JOIN payments pay
    ON o.id = pay.order_id
WHERE 
    o.status = 'delivered'
    AND pay.status = 'completed'
    AND pay.payment_type = 'payment'
GROUP BY 
    c.name
ORDER BY 
    margen DESC;""")

df = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

#### 3. TOP 10 PRODUCTOS MÁS VENDIDOS

In [ ]:
''' 

'''

cursor.execute("""
SELECT 
    p.id AS product_id,
    p.name AS producto,
    SUM(oi.quantity) AS unidades_vendidas
FROM orders o
INNER JOIN order_items oi 
    ON o.id = oi.order_id
INNER JOIN products p 
    ON oi.product_id = p.id
INNER JOIN payments pay
    ON o.id = pay.order_id
WHERE 
    o.status = 'delivered'
    AND pay.status = 'completed'
    AND pay.payment_type = 'payment'
GROUP BY 
    p.id, p.name
ORDER BY 
    unidades_vendidas DESC
LIMIT 10;""")

df = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

#### 4. CLIENTES POR PAÍS Y CANAL

In [ ]:
''' 

'''
cursor.execute("""
SELECT 
    country,
    source_channel,
    COUNT(*) AS total_clientes
FROM customers
GROUP BY 
    country,
    source_channel
ORDER BY 
    country,
    total_clientes DESC;""")

df = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

#### 5. RATING MEDIO POR CATEGORÍA O PRODUCTO

In [ ]:
''' 

'''
cursor.execute("""
SELECT 
    c.name AS categoria,
    AVG(r.rating) AS rating_medio
FROM reviews r
INNER JOIN order_items oi 
    ON r.order_item_id = oi.id
INNER JOIN products p 
    ON oi.product_id = p.id
INNER JOIN categories c
    ON p.category_id = c.id
GROUP BY 
    c.name
ORDER BY 
    rating_medio DESC;
""")

df = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])